# Explainability (SHAP + LIME)

This notebook:
1) runs the explanation pipeline (optional)
2) visualizes the generated CSV artifacts under `Results/<dataset>/<method>/explain/`

It does not require the `shap` Python package. It visualizes the **exported CSV summaries**.

In [ ]:
from pathlib import Path
import sys, os

REPO = Path().resolve()
assert (REPO/'qnm_qai.py').exists(), "Run Jupyter from the repository root (folder containing qnm_qai.py)"
print("Repo root:", REPO)
print("Python:", sys.executable)

# Best-effort: ensure Results exists
(Path("Results")).mkdir(exist_ok=True)


In [ ]:
ensure_matplotlib()
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score, balanced_accuracy_score

def _safe_div(num, den):
    return float(num) / float(den) if float(den) != 0.0 else float("nan")

def cm_metrics_from_preds(y_true, prob1, threshold=0.5):
    y_true = np.asarray(y_true, dtype=int)
    prob1 = np.asarray(prob1, dtype=float)
    y_pred = (prob1 >= float(threshold)).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sens = _safe_div(tp, tp + fn)
    spec = _safe_div(tn, tn + fp)
    ppv  = _safe_div(tp, tp + fp)
    npv  = _safe_div(tn, tn + fn)

    acc = accuracy_score(y_true, y_pred)
    bal = balanced_accuracy_score(y_true, y_pred)

    # AUC is undefined if only one class is present
    auc = float("nan")
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, prob1)

    return {
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "sensitivity": sens,
        "specificity": spec,
        "ppv": ppv,
        "npv": npv,
        "accuracy": float(acc),
        "balanced_accuracy": float(bal),
        "auc": float(auc),
    }, cm

def show_confusion_matrix(cm, title="Confusion matrix", labels=("0", "1")):
    import matplotlib.pyplot as plt
    import numpy as np

    cm = np.asarray(cm, dtype=int)
    fig, ax = plt.subplots(figsize=(4.2, 3.6))
    im = ax.imshow(cm)

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1], labels=labels)
    ax.set_yticks([0, 1], labels=labels)

    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center")

    fig.tight_layout()
    plt.show()

def show_roc_curve(y_true, prob1, title="ROC curve"):
    import matplotlib.pyplot as plt
    from sklearn.metrics import RocCurveDisplay

    if len(np.unique(y_true)) < 2:
        print("ROC: only one class present in y_true; skipping.")
        return
    RocCurveDisplay.from_predictions(y_true, prob1)
    plt.title(title)
    plt.show()

def ensure_matplotlib():
    try:
        import matplotlib.pyplot as _plt  # noqa: F401
    except Exception:
        # Notebook-safe install
        import sys
        !{sys.executable} -m pip install matplotlib


## Run SHAP/LIME on existing Results (slow)

This step can take a long time because the explainers call the model many times.

If you already ran explanations, skip this.

In [ ]:
!bash examples/run_explain_all.sh

## Visualize SHAP summary (mean |SHAP|)

Reads `shap_summary.csv` when present.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def plot_shap_summary(shap_csv: Path, top=20, title=None):
    import matplotlib.pyplot as plt

    df = pd.read_csv(shap_csv)
    # flexible column naming
    cand_cols = [c for c in df.columns if c.lower() in ("mean_abs_shap","mean_abs","abs_mean","mean_abs_value")]
    col = cand_cols[0] if cand_cols else df.columns[-1]

    df = df.sort_values(col, ascending=False).head(int(top)).copy()
    df = df.iloc[::-1]

    plt.figure(figsize=(6.5, 5.0))
    plt.barh(df["feature"], df[col])
    plt.title(title or f"SHAP summary — top {top}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

results = Path("Results")
shap_files = sorted(results.rglob("shap_summary.csv"))

print("Found SHAP summary files:", len(shap_files))
for p in shap_files[:10]:
    print(" -", p)

# Plot the first few
for p in shap_files[:3]:
    print("\n", p)
    display(pd.read_csv(p).head(10))
    plot_shap_summary(p, top=20, title=str(p.parent.parent))


## Visualize LIME explanations

Reads `lime_explanations.csv` when present.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def plot_lime_for_sample(lime_csv: Path, sample_id=None, top=15, title=None):
    import matplotlib.pyplot as plt

    df = pd.read_csv(lime_csv)
    # expected columns: sample_id, feature, weight
    if "sample_id" not in df.columns:
        # allow alternative naming
        for c in df.columns:
            if "sample" in c.lower() and "id" in c.lower():
                df = df.rename(columns={c: "sample_id"})
                break

    if sample_id is None:
        sample_id = df["sample_id"].iloc[0]

    d = df[df["sample_id"] == sample_id].copy()
    if d.empty:
        print("No rows for sample_id", sample_id)
        return

    # choose weight column
    wcol = "weight"
    if wcol not in d.columns:
        cand = [c for c in d.columns if "weight" in c.lower()]
        if cand:
            wcol = cand[0]
        else:
            wcol = d.columns[-1]

    # top by absolute weight
    d["absw"] = d[wcol].abs()
    d = d.sort_values("absw", ascending=False).head(int(top)).copy()
    d = d.sort_values(wcol, ascending=True)

    plt.figure(figsize=(6.5, 5.0))
    plt.barh(d["feature"], d[wcol])
    plt.title(title or f"LIME — sample {sample_id}")
    plt.xlabel(wcol)
    plt.tight_layout()
    plt.show()

results = Path("Results")
lime_files = sorted(results.rglob("lime_explanations.csv"))

print("Found LIME files:", len(lime_files))
for p in lime_files[:10]:
    print(" -", p)

for p in lime_files[:3]:
    print("\n", p)
    display(pd.read_csv(p).head(10))
    plot_lime_for_sample(p, sample_id=None, top=15, title=str(p.parent.parent))
